### SIG500 processing

#### Concatenating velocity files from OceanContour - Burst_*VTC.DSEL.AVER.nc

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import glob
import datetime

In [ ]:
folder="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/proc_1/rec_202508/BASS3A_PTSUVW_202308/SIG500_102455/Exported Data/S102455A007_BSS3A2408_0000/Burst"
os.chdir(folder)

fic1='Burst_001.VTC.DSEL.AVER.nc'
fic2='Burst_002.VTC.DSEL.AVER.nc'
filist = sorted(glob.glob("Burst_???.VTC.DSEL.AVER.nc"))


In [ ]:
list_ind = range(len(filist)-1)
list_ind
for ific in list_ind :
    file_in = filist[ific+1]
    print(file_in)


In [ ]:
# Open first file with decode_times=False to avoid timestamp overflow
file_in=filist[0]
ds = xr.open_dataset(file_in, group='Data/Burst', decode_times=False)

# Manually decode time with safe datetime conversion
if 'time' in ds.variables:
    # Convert time to float64 first, then to datetime64[s]
    time_vals = ds.time.values.astype('float64')
    # Assume time is in seconds since some epoch, convert to datetime64[s]
    time_decoded = pd.to_datetime(time_vals, unit='s', errors='coerce')
    ds = ds.assign_coords(time=('time', time_decoded))

list_ind = range(len(filist)-1)
for ific in list_ind :
    file_in = filist[ific+1]
    ds2 = xr.open_dataset(file_in, group='Data/Burst', decode_times=False)
    
    # Manually decode time for the second dataset
    if 'time' in ds2.variables:
        time_vals2 = ds2.time.values.astype('float64')
        time_decoded2 = pd.to_datetime(time_vals2, unit='s', errors='coerce')
        ds2 = ds2.assign_coords(time=('time', time_decoded2))
    
    # Use concat instead of merge for time series data
    ds = xr.concat([ds, ds2], dim='time')


In [ ]:
ds

In [ ]:
# plt.plot(ds.time,ds.BurstVelocityENU_Range)

In [ ]:
output_folder = "/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/proc_2/rec_202508/BASS3A_PTSUVW_202308/SIG500_102455"
output_file = os.path.join(output_folder, "BASS3A_V_202508.nc")
ds.to_netcdf(output_file)

In [ ]:
output_folder = "/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/proc_2/rec_202508/BASS3A_PTSUVW_202308/SIG500_102455"
os.chdir(output_folder)
file_in = "BASS3A_V_202508.nc"
# file_in = "merged_burst_waves__SWOTMID_JAS_2023_FSP.nc"
ds = xr.open_dataset(file_in)

In [ ]:
# start_plot = datetime.datetime(2023, 2, 14, 00)
start_plot = datetime.datetime(2024, 7, 31, 00)
# end_plot = datetime.datetime(2023, 8, 6, 00)
end_plot = datetime.datetime(2025, 8, 23, 00)
dsc=ds.sel(time=slice(start_plot, end_plot))

In [ ]:
plt.figure()
# plt.plot(ds.time[start_p:end_p],ds.Height_Hm0[start_p:end_p])
# plt.plot(dsc.time,dsc.Height_H3)
plt.plot(dsc.time,dsc.Altimeter_AST)

# plt.xlim([start_plot, end_plot])